In [ ]:
'''
The actual file where I train the models
'''

In [7]:
'''
import ML libraries
'''
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import os

In [8]:
def merge_and_save_data(text_parquet: str, labels_csv: str, output_parquet: str):
    df_labels = pd.read_csv(labels_csv)
    
    valid_tickers = df_labels['tickers'].unique()
    
    df_text = pd.read_parquet(text_parquet)
    
    # standardize dates to match
    df_text['date_str'] = pd.to_datetime(df_text['datetime']).dt.strftime('%Y-%m-%d')
    df_text = df_text.explode('tickers').dropna(subset=['tickers'])
    
    df_text = df_text[df_text['tickers'].isin(valid_tickers)]
    
    print("Merging")
    master_df = pd.merge(df_text, df_labels, how='inner', on=['tickers', 'date_str'])
    
    master_df = master_df.dropna(subset=['clean_text', 'label'])
    
    print(f"Training Dataset Ready: {len(master_df):,} labeled posts.")
    print(f"saving to {output_parquet}...")
    master_df.to_parquet(output_parquet, engine='pyarrow', index=False)
    print("merge complete")

if __name__ == "__main__":
    TEXT_DATA = "wsb_submissions_ml_ready.parquet"
    LABEL_DATA = "market_labels.csv" 
    OUTPUT_FILE = "ml_training_data.parquet"
    
    if os.path.exists(TEXT_DATA) and os.path.exists(LABEL_DATA):
        merge_and_save_data(TEXT_DATA, LABEL_DATA, OUTPUT_FILE)
    else:
        print("cannot find the parquet or CSV files")

Merging
Training Dataset Ready: 75,133 labeled posts.
saving to ml_training_data.parquet...
merge complete


In [10]:
def get_and_save_eighty_twenty_split_training_and_testing_parquet(master_training_parquet, training_parquet, testing_parquet, split_ratio: float = 0.8):
    """
    Chronologically splits the master dataset and saves them as immutable Parquet artifacts.
    This guarantees no data leakage across different ML experiments.
    """
    master_training_df = pd.read_parquet(master_training_parquet)
    master_training_df = master_training_df.sort_values('datetime')
    split_index = int(len(master_training_df) * split_ratio)
    train_df = master_training_df.iloc[:split_index].copy()
    test_df = master_training_df.iloc[split_index:].copy()
    print('Split complete')
    print(f'\n Check dataframe lengths: training -> {len(train_df)}, testing -> {len(test_df)}')
    print(f"\n Training Timeline: {train_df['datetime'].min()} to {train_df['datetime'].max()}")
    print(f"\n Testing Timeline: {test_df['datetime'].min()} to {test_df['datetime'].max()}")
    train_df.to_parquet(training_parquet, engine='pyarrow', index=False) # always false to prevent numbered rows
    test_df.to_parquet(testing_parquet, engine='pyarrow', index=False)
    print('80/20 training and testing data saved')

if __name__ == "__main__":
    MASTER_TRAINING_PARQUET = 'ml_training_data.parquet'
    TRAINING_PARQUET_NAME = 'eighty_split_ml_training_data.parquet'
    TESTING_PARQUET = 'twenty_split_ml_testing_data.parquet'

    if os.path.exists(MASTER_TRAINING_PARQUET):
        get_and_save_eighty_twenty_split_training_and_testing_parquet(MASTER_TRAINING_PARQUET, TRAINING_PARQUET_NAME, TESTING_PARQUET)
    else:
        print("cannot find the parquet")

Split complete

 Check dataframe lengths: training -> 60106, testing -> 15027

 Training Timeline: 2012-04-11 16:40:40 to 2022-12-27 22:15:41

 Testing Timeline: 2022-12-27 23:17:49 to 2025-12-31 19:23:05
80/20 training and testing data saved


In [11]:
'''
Logistic Regression training cell
'''
def train_and_evaluate_logistic_regression(X_train, y_train, X_test, y_test, vectorizer):
    '''
    Trains, evaluates, and extracts interpretable features for Logistic Regression
    '''
    # class_weight='balanced' forces the model to care equally about UP and DOWN days
    lr_model = LogisticRegression(max_iter=1000, class_weight='balanced')
    lr_model.fit(X_train, y_train)
    lr_preds = lr_model.predict(X_test)
    
    print("Logistic Regression Testing Results")
    print(f"\n Accuracy: {accuracy_score(y_test, lr_preds):.2%}")
    print(classification_report(y_test, lr_preds))
    
    print("\n TOP BULLISH INDICATOR WORDS (from LR model):")
    feature_names = vectorizer.get_feature_names_out()
    coefficients = lr_model.coef_[0]
    
    word_weights = pd.DataFrame({'Word': feature_names, 'Weight': coefficients})
    print(word_weights.sort_values('Weight', ascending=False).head(10))
    
    print("\n TOP BEARISH INDICATOR WORDS:")
    print(word_weights.sort_values('Weight', ascending=True).head(10))
    
    return lr_model

def run_LG_training(train_parquet: str, test_parquet: str):
    train_df = pd.read_parquet(train_parquet)
    test_df = pd.read_parquet(test_parquet)
    print('Checking training and testing data timeline')
    print(f"\nTraining on: {train_df['datetime'].min()} to {train_df['datetime'].max()} ({len(train_df):,} posts)")
    print(f"Testing on:  {test_df['datetime'].min()} to {test_df['datetime'].max()} ({len(test_df):,} posts)")

    majority_class_ratio = test_df['label'].value_counts(normalize=True).max()
    print(f'Baseline accuracy check for testing: {majority_class_ratio:.2%}')
    print("\nVectorizing Text")
    vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
    
    # fit the vocabulary strictly on the training data only
    X_train = vectorizer.fit_transform(train_df['clean_text'])
    y_train = train_df['label']
    
    # test data 
    X_test = vectorizer.transform(test_df['clean_text'])
    y_test = test_df['label']

    lr_model = train_and_evaluate_logistic_regression(X_train, y_train, X_test, y_test, vectorizer)
    return lr_model

if __name__ == "__main__":
    TRAIN_DATA = 'eighty_split_ml_training_data.parquet'
    TEST_DATA = 'twenty_split_ml_testing_data.parquet'
    
    if os.path.exists(TRAIN_DATA) and os.path.exists(TEST_DATA):
        print("Logistic Regression Pipeline is ready")
        run_LG_training(TRAIN_DATA, TEST_DATA)
    else:
        print("Cannot find the train/test artifacts")

Logistic Regression Pipeline is ready
Checking training and testing data timeline

Training on: 2012-04-11 16:40:40 to 2022-12-27 22:15:41 (60,106 posts)
Testing on:  2022-12-27 23:17:49 to 2025-12-31 19:23:05 (15,027 posts)
Baseline accuracy check: 52.19%

Vectorizing Text
Logistic Regression Testing Results

 Accuracy: 50.38%
              precision    recall  f1-score   support

           0       0.48      0.46      0.47      7185
           1       0.52      0.54      0.53      7842

    accuracy                           0.50     15027
   macro avg       0.50      0.50      0.50     15027
weighted avg       0.50      0.50      0.50     15027


 TOP BULLISH INDICATOR WORDS (from LR model):
            Word    Weight
560          bot  2.613077
1465    finished  1.957099
2424      mainly  1.837906
310    april put  1.820227
981      data is  1.725575
838   company is  1.695560
2856       of to  1.694935
2514     medical  1.593625
1670      get to  1.538650
430         base  1.490739

In [14]:
'''
Random Forest training cell
'''
def train_and_evaluate_random_forest(X_train, y_train, X_test, y_test, vectorizer):
    """Trains and evaluates the Non-Linear Random Forest classifier."""
    print("\n Training Random Forest")
    rf_model = RandomForestClassifier(n_estimators=100, max_depth=15, class_weight='balanced', random_state=42, n_jobs=-1)
    rf_model.fit(X_train, y_train)
    rf_preds = rf_model.predict(X_test)
    
    print("RANDOM FOREST RESULTS")
    print(f"Accuracy: {accuracy_score(y_test, rf_preds):.2%}")
    print(classification_report(y_test, rf_preds))

    print("\n most important words (from Random Forest):")
    feature_names = vectorizer.get_feature_names_out()
    importances = rf_model.feature_importances_
    
    word_weights = pd.DataFrame({'Word': feature_names, 'Importance': importances})
    print(word_weights.sort_values('Importance', ascending=False).head(10))
    
    return rf_model

def run_RF_training(train_parquet: str, test_parquet: str):
    train_df = pd.read_parquet(train_parquet)
    test_df = pd.read_parquet(test_parquet)
    print('Checking training and testing data timeline')
    print(f"\nTraining on: {train_df['datetime'].min()} to {train_df['datetime'].max()} ({len(train_df):,} posts)")
    print(f"Testing on:  {test_df['datetime'].min()} to {test_df['datetime'].max()} ({len(test_df):,} posts)")

    majority_class_ratio = test_df['label'].value_counts(normalize=True).max()
    print(f'Baseline accuracy check for testing: {majority_class_ratio:.2%}')
    print("\nVectorizing Text")
    vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
    
    X_train = vectorizer.fit_transform(train_df['clean_text'])
    y_train = train_df['label']
    
    X_test = vectorizer.transform(test_df['clean_text'])
    y_test = test_df['label']

    rf_model = train_and_evaluate_random_forest(X_train, y_train, X_test, y_test, vectorizer)
    return rf_model

if __name__ == "__main__":
    TRAIN_DATA = 'eighty_split_ml_training_data.parquet'
    TEST_DATA = 'twenty_split_ml_testing_data.parquet'
    
    if os.path.exists(TRAIN_DATA) and os.path.exists(TEST_DATA):
        print("Random Forest Pipeline is ready")
        run_RF_training(TRAIN_DATA, TEST_DATA)
    else:
        print("Cannot find the train/test artifacts")

Random Forest Pipeline is ready
Checking training and testing data timeline

Training on: 2012-04-11 16:40:40 to 2022-12-27 22:15:41 (60,106 posts)
Testing on:  2022-12-27 23:17:49 to 2025-12-31 19:23:05 (15,027 posts)
Baseline accuracy check for testing: 52.19%

Vectorizing Text

 Training Random Forest
RANDOM FOREST RESULTS
Accuracy: 50.10%
              precision    recall  f1-score   support

           0       0.48      0.44      0.46      7185
           1       0.52      0.56      0.54      7842

    accuracy                           0.50     15027
   macro avg       0.50      0.50      0.50     15027
weighted avg       0.50      0.50      0.50     15027


 most important words (from Random Forest):
              Word  Importance
3498     robinhood    0.021007
1682           gme    0.010135
440           bbby    0.007651
560            bot    0.004302
3484            rh    0.004208
2447  manipulation    0.003913
626            buy    0.003616
124       allowing    0.003482
4233

In [21]:
'''
Grid Search run overnight Random Forest cell
'''
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, accuracy_score
import joblib
import os

def run_overnight_grid_search(train_parquet: str, test_parquet: str):
    print("the Grid Search Optimizer is on an overnight run so ima go mimimimi")
    
    train_df = pd.read_parquet(train_parquet)
    test_df = pd.read_parquet(test_parquet)
    
    X_train = train_df['clean_text']
    y_train = train_df['label']
    X_test = test_df['clean_text']
    y_test = test_df['label']

    # this pipeline chains the Vectorizer and the Model together so GridSearch can tweak both simultaneously
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(stop_words='english')),
        ('rf', RandomForestClassifier(class_weight='balanced', random_state=42))
    ])

    # matrix creates 3 x 3 x 3 x 2 = 54 different models.
    # will train 162 total models with 3-fold cross-validation, 
    param_grid = {
        # test different vocabulary sizes and n-grams
        'tfidf__max_features': [3000, 5000, 10000],
        'tfidf__ngram_range': [(1, 1), (1, 2), (1, 3)],
        
        # Test how deep the trees should go (None -> infinite depth for memorizing patterns)
        'rf__max_depth': [15, 30, None],
        
        # Test how many trees to build
        'rf__n_estimators': [100, 300]
    }

    print("\nconfiguring grid search matrix...")
    grid_search = GridSearchCV(
        pipeline, 
        param_grid, 
        cv=3,                 # 3-Fold Cross Validation 
        scoring='f1',         # Optimize specifically for a balanced F1-Score, not just raw accuracy
        n_jobs=-1,            # Use 100% of CPU cores
        verbose=3             # prints live updates to the terminal
    )

    print("\n starting training loop")
    grid_search.fit(X_train, y_train)
    print("finished")
    
    
    best_params = grid_search.best_params_
    print("\n WINNING PARAMETERS HOLY SHIT:")
    for key, value in best_params.items():
        print(f"{key}: {value}")

    # The grid_search object automatically saves the #1 best model it found
    best_model = grid_search.best_estimator_
    
    # test winning model against blind test set (2023-2025)
    print("\n winning model evaluation:")
    final_preds = best_model.predict(X_test)
    
    print(f"Accuracy: {accuracy_score(y_test, final_preds):.2%}")
    print(classification_report(y_test, final_preds))

    save_path = "best_rf_pipeline.pkl"
    joblib.dump(best_model, save_path)
    print(f"\nsaved the winning model to {save_path}")

if __name__ == "__main__":
    TRAIN_DATA = 'eighty_split_ml_training_data.parquet'
    TEST_DATA = 'twenty_split_ml_testing_data.parquet'
    
    if os.path.exists(TRAIN_DATA) and os.path.exists(TEST_DATA):
        run_overnight_grid_search(TRAIN_DATA, TEST_DATA)
    else:
        print("Cannot find the train/test artifacts.")

the Grid Search Optimizer is on an overnight run so ima go mimimimi

configuring grid search matrix...

 starting training loop
Fitting 3 folds for each of 54 candidates, totalling 162 fits
[CV 2/3] END rf__max_depth=15, rf__n_estimators=100, tfidf__max_features=3000, tfidf__ngram_range=(1, 1);, score=0.374 total time=  53.2s
[CV 1/3] END rf__max_depth=15, rf__n_estimators=100, tfidf__max_features=3000, tfidf__ngram_range=(1, 1);, score=0.581 total time=  55.3s
[CV 2/3] END rf__max_depth=15, rf__n_estimators=100, tfidf__max_features=5000, tfidf__ngram_range=(1, 1);, score=0.359 total time=  57.5s
[CV 1/3] END rf__max_depth=15, rf__n_estimators=100, tfidf__max_features=5000, tfidf__ngram_range=(1, 1);, score=0.555 total time=  58.6s
[CV 3/3] END rf__max_depth=15, rf__n_estimators=100, tfidf__max_features=3000, tfidf__ngram_range=(1, 1);, score=0.452 total time=  59.5s
[CV 2/3] END rf__max_depth=15, rf__n_estimators=100, tfidf__max_features=3000, tfidf__ngram_range=(1, 2);, score=0.367 t

/opt/anaconda3/envs/quant_club/lib/python3.9/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[CV 3/3] END rf__max_depth=None, rf__n_estimators=300, tfidf__max_features=3000, tfidf__ngram_range=(1, 1);, score=0.477 total time=44.8min
[CV 1/3] END rf__max_depth=None, rf__n_estimators=300, tfidf__max_features=3000, tfidf__ngram_range=(1, 2);, score=0.580 total time=50.4min
[CV 2/3] END rf__max_depth=None, rf__n_estimators=300, tfidf__max_features=3000, tfidf__ngram_range=(1, 2);, score=0.410 total time=51.0min
[CV 3/3] END rf__max_depth=None, rf__n_estimators=300, tfidf__max_features=3000, tfidf__ngram_range=(1, 2);, score=0.471 total time=46.2min
[CV 3/3] END rf__max_depth=None, rf__n_estimators=300, tfidf__max_features=3000, tfidf__ngram_range=(1, 3);, score=0.469 total time=45.4min
[CV 1/3] END rf__max_depth=None, rf__n_estimators=300, tfidf__max_features=3000, tfidf__ngram_range=(1, 3);, score=0.576 total time=49.5min
[CV 2/3] END rf__max_depth=None, rf__n_estimators=300, tfidf__max_features=3000, tfidf__ngram_range=(1, 3);, score=0.403 total time=49.6min
[CV 1/3] END rf__max

In [20]:
'''
Naive-Bayes training cell
'''

import pandas as pd
import numpy as np
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics import accuracy_score, classification_report
import os

def train_and_evaluate_naive_bayes(X_train, y_train, X_test, y_test, vectorizer):
    """Trains and evaluates the Multinomial Naive Bayes classifier."""
    print("\n Training Naive Bayes")
    # alpha=0.5 applies slight smoothing to handle words the model hasn't seen before
    nb_model = MultinomialNB(alpha=0.5)
    nb_model.fit(X_train, y_train)
    nb_preds = nb_model.predict(X_test)
    
    print("NAIVE BAYES RESULTS")
    print(f"Accuracy: {accuracy_score(y_test, nb_preds):.2%}")
    print(classification_report(y_test, nb_preds))
    
    # words that have the highest probability of being bullish or bearish
    print("\nMOST PROBABLE WORDS (From Naive Bayes):")
    feature_names = vectorizer.get_feature_names_out()
    
    log_prob_diff = nb_model.feature_log_prob_[1] - nb_model.feature_log_prob_[0]
    
    word_weights = pd.DataFrame({'Word': feature_names, 'Log_Prob_Difference': log_prob_diff})
    
    print("\nTop Bullish Words (Highest positive probability shift):")
    print(word_weights.sort_values('Log_Prob_Difference', ascending=False).head(10))
    
    print("\nTop Bearish Words (Highest negative probability shift):")
    print(word_weights.sort_values('Log_Prob_Difference', ascending=True).head(10))
    
    return nb_model

def run_NB_training(train_parquet: str, test_parquet: str):
    train_df = pd.read_parquet(train_parquet)
    test_df = pd.read_parquet(test_parquet)
    
    print('Checking training and testing data timeline')
    print(f"\nTraining on: {train_df['datetime'].min()} to {train_df['datetime'].max()} ({len(train_df):,} posts)")
    print(f"Testing on:  {test_df['datetime'].min()} to {test_df['datetime'].max()} ({len(test_df):,} posts)")

    majority_class_ratio = test_df['label'].value_counts(normalize=True).max()
    print(f'Baseline accuracy check for testing: {majority_class_ratio:.2%}')
    
    print("\nVectorizing Text")
    calendar_noise = [
        'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday',
        'mon', 'tues', 'wed', 'thurs', 'fri', 'sat', 'sun',
        'january', 'february', 'march', 'april', 'may', 'june', 'july', 'august',
        'september', 'october', 'november', 'december',
        'jan', 'feb', 'mar', 'apr', 'jun', 'jul', 'aug', 'sep', 'sept', 'oct', 'nov', 'dec'
    ]
    custom_stop_words = list(ENGLISH_STOP_WORDS.union(calendar_noise))

    # custom stop words added due to model being lazy and memorizing dates and days of the week 
    vectorizer = TfidfVectorizer(
        max_features=5000, 
        ngram_range=(1, 2), 
        stop_words=custom_stop_words
    )
    
    X_train = vectorizer.fit_transform(train_df['clean_text'])
    y_train = train_df['label']
    
    X_test = vectorizer.transform(test_df['clean_text'])
    y_test = test_df['label']

    nb_model = train_and_evaluate_naive_bayes(X_train, y_train, X_test, y_test, vectorizer)
    return nb_model

if __name__ == "__main__":
    TRAIN_DATA = 'eighty_split_ml_training_data.parquet'
    TEST_DATA = 'twenty_split_ml_testing_data.parquet'
    
    if os.path.exists(TRAIN_DATA) and os.path.exists(TEST_DATA):
        print("Naive Bayes Pipeline is ready")
        run_NB_training(TRAIN_DATA, TEST_DATA)
    else:
        print("cant find the train/test artifacts")

Naive Bayes Pipeline is ready
Checking training and testing data timeline

Training on: 2012-04-11 16:40:40 to 2022-12-27 22:15:41 (60,106 posts)
Testing on:  2022-12-27 23:17:49 to 2025-12-31 19:23:05 (15,027 posts)
Baseline accuracy check for testing: 52.19%

Vectorizing Text

 Training Naive Bayes
NAIVE BAYES RESULTS
Accuracy: 49.14%
              precision    recall  f1-score   support

           0       0.48      0.72      0.58      7185
           1       0.52      0.28      0.36      7842

    accuracy                           0.49     15027
   macro avg       0.50      0.50      0.47     15027
weighted avg       0.50      0.49      0.47     15027


MOST PROBABLE WORDS (From Naive Bayes):

Top Bullish Words (Highest positive probability shift):
                      Word  Log_Prob_Difference
4002           share stake             1.279029
3030             nonmarket             0.949124
1876             fund flow             0.947088
180          amet eurozone             0.856

In [ ]:
'''
Support Vector Machine training cell
'''
def train_and_evaluate_svm(X_train, y_train, X_test, y_test):
    """Trains and evaluates a Linear Support Vector Machine (Highly optimized for Text/TF-IDF)."""
    print("\nTraining Support Vector Machine (LinearSVC)...")
    # LinearSVC is incredibly fast for high-dimensional sparse data like TF-IDF
    svm_model = LinearSVC(class_weight='balanced', random_state=42, max_iter=2000)
    svm_model.fit(X_train, y_train)
    svm_preds = svm_model.predict(X_test)
    
    print("\n" + "="*40)
    print("⚔️ SUPPORT VECTOR MACHINE RESULTS")
    print("="*40)
    print(f"Accuracy: {accuracy_score(y_test, svm_preds):.2%}")
    print(classification_report(y_test, svm_preds))
    
    return svm_model